# Neural Network Fundamentals

A comprehensive introduction to neural networks from scratch.

## Learning Objectives

- Understand the structure of neural networks
- Implement forward propagation from scratch
- Understand activation functions
- Implement backpropagation from scratch
- Build a simple neural network without frameworks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

## 1. The Perceptron: Building Block

**The perceptron is the fundamental unit of neural networks - a simplified model of a biological neuron.**

### The Mathematical Model:
$$y = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$$

Where:
- $\mathbf{x}$ = input vector (features)
- $\mathbf{w}$ = weight vector (learned parameters)

- $b$ = bias term (threshold adjustment)Activation functions let networks learn **any function** (universal approximation theorem).

- $\sigma$ = activation function (introduces non-linearity)

$$W_1 \cdot W_2 \cdot ... \cdot W_{100} \cdot x = W_{equivalent} \cdot x$$

### Biological Inspiration:Without activation functions, a 100-layer network is equivalent to a single linear layer:

### Why Non-Linearity Matters:

| Biological Neuron | Artificial Perceptron |

|-------------------|----------------------|| **Activation** | Non-linear transformation | Without it, network = linear regression! |

| Dendrites | Inputs ($x_i$) || **Bias** | Shift activation threshold | Allows activation even with zero input |

| Synapse strength | Weights ($w_i$) || **Weights** | Scale input importance | Learned from data via gradient descent |

| Cell body (soma) | Weighted sum + bias ||-----------|---------|------------------|

| Axon firing | Activation function || Component | Purpose | ML Interpretation |


### What Each Component Does:

In [ ]:
# Simple perceptron visualization
def visualize_perceptron():
    fig, ax = plt.subplots(figsize=(10, 6))

    # Input nodes
    inputs = ["x₁", "x₂", "x₃"]
    for i, inp in enumerate(inputs):
        circle = plt.Circle((0.2, 0.7 - i * 0.2), 0.08, color="lightblue", ec="black")
        ax.add_patch(circle)
        ax.text(0.2, 0.7 - i * 0.2, inp, ha="center", va="center", fontsize=12)

    # Weights
    for i in range(3):
        ax.annotate(
            "",
            xy=(0.5, 0.5),
            xytext=(0.28, 0.7 - i * 0.2),
            arrowprops=dict(arrowstyle="->", color="gray"),
        )
        ax.text(0.35, 0.6 - i * 0.15, f"w{i+1}", fontsize=10, color="gray")

    # Summation node
    circle = plt.Circle((0.5, 0.5), 0.1, color="lightyellow", ec="black")
    ax.add_patch(circle)
    ax.text(0.5, 0.5, "Σ", ha="center", va="center", fontsize=14)

    # Activation
    ax.annotate(
        "",
        xy=(0.7, 0.5),
        xytext=(0.6, 0.5),
        arrowprops=dict(arrowstyle="->", color="gray"),
    )
    circle = plt.Circle((0.7, 0.5), 0.08, color="lightgreen", ec="black")
    ax.add_patch(circle)
    ax.text(0.7, 0.5, "σ", ha="center", va="center", fontsize=14)

    # Output
    ax.annotate(
        "",
        xy=(0.9, 0.5),
        xytext=(0.78, 0.5),
        arrowprops=dict(arrowstyle="->", color="gray"),
    )
    ax.text(0.9, 0.5, "y", ha="center", va="center", fontsize=14)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("Single Perceptron Structure", fontsize=14)
    plt.show()


visualize_perceptron()

## 2. Activation Functions

**Activation functions introduce non-linearity, enabling networks to learn complex patterns.**

| Regression output | Linear (no activation) | Unrestricted range |

### The Activation Function Zoo:| Multi-class output | Softmax | Outputs probability distribution |

| Binary output | Sigmoid | Outputs probability in (0,1) |

| Function | Formula | Range | Use Case || Hidden layers | ReLU (or variants) | Fast, no vanishing gradient |

|----------|---------|-------|----------||-------|-------------|-----|

| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | (0, 1) | Output layer (binary classification) || Layer | Recommended | Why |

| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | (-1, 1) | Hidden layers (centered output) |

| **ReLU** | $\max(0, x)$ | [0, ∞) | Most hidden layers (default choice!) |### Choosing Activation Functions:

| **Leaky ReLU** | $\max(0.01x, x)$ | (-∞, ∞) | Avoid "dying ReLU" |

| **Softmax** | $\frac{e^{x_i}}{\sum e^{x_j}}$ | (0, 1), sums to 1 | Output (multi-class) |- Deep networks couldn't learn before ReLU

- Each layer multiplies: $0.25^n$ becomes tiny!

### Why ReLU Dominates Modern Deep Learning:- Sigmoid: max derivative = 0.25 (at x=0)

Sigmoid/Tanh derivatives are small when inputs are large:

1. **Computational efficiency**: Just a max operation

2. **Sparse activation**: Many neurons output 0 (efficient)### The Vanishing Gradient Problem:

3. **No vanishing gradient**: Gradient is 1 for positive inputs
4. **Biological plausibility**: Similar to neuron firing patterns

In [ ]:
# Activation functions
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)


def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(float)


def tanh(x):
    return np.tanh(x)


def tanh_derivative(x):
    return 1 - np.tanh(x) ** 2


def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)


def leaky_relu_derivative(x, alpha=0.01):
    return np.where(x > 0, 1, alpha)


# Visualize activation functions
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Activation functions
activations = [
    ("Sigmoid", sigmoid, sigmoid_derivative),
    ("Tanh", tanh, tanh_derivative),
    ("ReLU", relu, relu_derivative),
    ("Leaky ReLU", leaky_relu, leaky_relu_derivative),
]

for i, (name, func, deriv) in enumerate(activations):
    axes[0, i].plot(x, func(x), "b-", linewidth=2)
    axes[0, i].axhline(y=0, color="k", linewidth=0.5)
    axes[0, i].axvline(x=0, color="k", linewidth=0.5)
    axes[0, i].set_title(f"{name}")
    axes[0, i].set_xlabel("x")
    axes[0, i].set_ylabel("f(x)")
    axes[0, i].grid(True)

    axes[1, i].plot(x, deriv(x), "r-", linewidth=2)
    axes[1, i].axhline(y=0, color="k", linewidth=0.5)
    axes[1, i].axvline(x=0, color="k", linewidth=0.5)
    axes[1, i].set_title(f"{name} Derivative")
    axes[1, i].set_xlabel("x")
    axes[1, i].set_ylabel("f'(x)")
    axes[1, i].grid(True)

plt.tight_layout()
plt.show()

## 3. Neural Network from Scratch

### Forward Propagation: How Predictions Are Made

For each layer $l$:

These are needed for backpropagation!

**Step 1 - Linear transformation (weighted sum):**During forward pass, we store $z$ and $a$ values at each layer.

$$\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$### Why We Cache Values:



**Step 2 - Non-linear activation:**```

$$\mathbf{a}^{[l]} = \sigma(\mathbf{z}^{[l]})$$b1: [4, 1]    b2: [1, 1]

W1: [4, 2]    W2: [1, 4]

### The Matrix Dimensions:

     [2, m]              [4, m]              [1, m]

| Variable | Dimensions | Meaning |Input (2 features) → Hidden (4 neurons) → Output (1 neuron)

|----------|------------|--------|```

| $\mathbf{W}^{[l]}$ | (neurons in l, neurons in l-1) | Weights connecting layers |

| $\mathbf{b}^{[l]}$ | (neurons in l, 1) | Bias for each neuron |### Example: 3-Layer Network [2, 4, 1]

| $\mathbf{a}^{[l]}$ | (neurons in l, batch_size) | Activations (outputs) |

In [ ]:
class NeuralNetwork:
    """A simple feedforward neural network from scratch."""

    def __init__(self, layer_sizes, activation="relu", learning_rate=0.01):
        """
        Initialize neural network.

        Parameters
        ----------
        layer_sizes : list
            List of layer sizes [input, hidden1, hidden2, ..., output]
        activation : str
            Activation function: 'relu', 'sigmoid', or 'tanh'
        learning_rate : float
            Learning rate for gradient descent
        """
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.num_layers = len(layer_sizes)

        # Set activation function
        if activation == "relu":
            self.activation = relu
            self.activation_derivative = relu_derivative
        elif activation == "sigmoid":
            self.activation = sigmoid
            self.activation_derivative = sigmoid_derivative
        else:
            self.activation = tanh
            self.activation_derivative = tanh_derivative

        # Initialize weights and biases with He initialization
        self.weights = []
        self.biases = []

        for i in range(1, self.num_layers):
            w = np.random.randn(layer_sizes[i], layer_sizes[i - 1]) * np.sqrt(
                2.0 / layer_sizes[i - 1]
            )
            b = np.zeros((layer_sizes[i], 1))
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, X):
        """Forward propagation."""
        self.activations = [X.T]  # Store for backprop
        self.z_values = []  # Pre-activation values

        a = X.T
        for i in range(len(self.weights) - 1):
            z = self.weights[i] @ a + self.biases[i]
            self.z_values.append(z)
            a = self.activation(z)
            self.activations.append(a)

        # Output layer (sigmoid for binary classification)
        z = self.weights[-1] @ a + self.biases[-1]
        self.z_values.append(z)
        a = sigmoid(z)
        self.activations.append(a)

        return a.T

    def backward(self, X, y):
        """Backpropagation."""
        m = X.shape[0]
        y = y.reshape(-1, 1).T

        # Compute gradients
        gradients_w = []
        gradients_b = []

        # Output layer gradient
        dz = self.activations[-1] - y  # Derivative of cross-entropy + sigmoid
        dw = (1 / m) * dz @ self.activations[-2].T
        db = (1 / m) * np.sum(dz, axis=1, keepdims=True)
        gradients_w.insert(0, dw)
        gradients_b.insert(0, db)

        # Hidden layers
        for i in range(len(self.weights) - 2, -1, -1):
            dz = (self.weights[i + 1].T @ dz) * self.activation_derivative(
                self.z_values[i]
            )
            dw = (1 / m) * dz @ self.activations[i].T
            db = (1 / m) * np.sum(dz, axis=1, keepdims=True)
            gradients_w.insert(0, dw)
            gradients_b.insert(0, db)

        # Update weights
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * gradients_w[i]
            self.biases[i] -= self.learning_rate * gradients_b[i]

    def compute_loss(self, y_pred, y_true):
        """Binary cross-entropy loss."""
        m = len(y_true)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = -(1 / m) * np.sum(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )
        return loss

    def fit(self, X, y, epochs=1000, verbose=True):
        """Train the network."""
        self.loss_history = []

        for epoch in range(epochs):
            # Forward pass
            y_pred = self.forward(X)

            # Compute loss
            loss = self.compute_loss(y_pred, y)
            self.loss_history.append(loss)

            # Backward pass
            self.backward(X, y)

            if verbose and epoch % 100 == 0:
                accuracy = np.mean((y_pred.flatten() > 0.5) == y)
                print(f"Epoch {epoch}: Loss = {loss:.4f}, Accuracy = {accuracy:.4f}")

    def predict(self, X):
        """Make predictions."""
        return (self.forward(X).flatten() > 0.5).astype(int)


print("NeuralNetwork class defined successfully!")

## 4. Training on Moon Dataset

In [ ]:
# Generate dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train network
nn = NeuralNetwork(layer_sizes=[2, 16, 8, 1], activation="relu", learning_rate=0.1)
nn.fit(X_train_scaled, y_train, epochs=1000)

In [ ]:
# Evaluate
y_pred_train = nn.predict(X_train_scaled)
y_pred_test = nn.predict(X_test_scaled)

train_acc = np.mean(y_pred_train == y_train)
test_acc = np.mean(y_pred_test == y_test)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Visualize decision boundary
def plot_decision_boundary(model, X, y, scaler=None, title="Decision Boundary"):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

    grid = np.c_[xx.ravel(), yy.ravel()]
    if scaler:
        grid = scaler.transform(grid)

    Z = model.forward(grid).reshape(xx.shape)

    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, levels=50, cmap="RdYlBu", alpha=0.8)
    plt.colorbar(label="Probability")
    plt.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=2)
    plt.scatter(
        X[y == 0, 0], X[y == 0, 1], c="blue", edgecolors="black", label="Class 0"
    )
    plt.scatter(
        X[y == 1, 0], X[y == 1, 1], c="red", edgecolors="black", label="Class 1"
    )
    plt.legend()
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()


plot_decision_boundary(nn, X_train, y_train, scaler, "Neural Network Decision Boundary")

In [ ]:
# Plot loss curve
plt.figure(figsize=(10, 5))
plt.plot(nn.loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over Time")
plt.grid(True)
plt.show()

## 5. Understanding Backpropagation

**Backpropagation is just the chain rule applied systematically!**

### The Goal:
Find $\frac{\partial \mathcal{L}}{\partial W}$ and $\frac{\partial \mathcal{L}}{\partial b}$ to update weights.


### The Chain Rule:- Gradient clipping

For a network with layers $L_1, L_2, \ldots, L_n$:- Residual connections (skip connections)

- Batch normalization

$$\frac{\partial \mathcal{L}}{\partial W^{[l]}} = \frac{\partial \mathcal{L}}{\partial a^{[L]}} \cdot \frac{\partial a^{[L]}}{\partial z^{[L]}} \cdot \frac{\partial z^{[L]}}{\partial a^{[L-1]}} \cdots \frac{\partial z^{[l]}}{\partial W^{[l]}}$$- ReLU activation (gradient = 1 for positive inputs)

**Solutions:**

### Step-by-Step Intuition:

- Large gradients → Exploding: training becomes unstable

1. **Output layer**: Compare prediction to true label (compute loss gradient)- Small gradients (σ, tanh) → Vanishing: early layers don't learn

2. **Propagate backward**: How much did each earlier layer contribute to error?During backprop, gradients are MULTIPLIED through layers:

3. **Update weights**: Move in direction that reduces loss### The Vanishing Gradient Problem Revisited:



### Key Gradients:| Linear | $dz/db = 1$ | Always 1 |

| Linear | $dz/dW = a^{[l-1]}$ | Previous activation |

| Layer Component | Gradient | Formula || Activation | $da/dz$ | Activation derivative |

|-----------------|----------|--------|| Loss → Output | $dL/da$ | Depends on loss function |

In [ ]:
# Visualize gradients flowing through the network
def visualize_backprop():
    fig, ax = plt.subplots(figsize=(14, 8))

    # Layer positions
    layers = [3, 4, 4, 2]  # Nodes per layer
    layer_positions = [0.1, 0.35, 0.6, 0.85]

    # Draw nodes
    for l, (n_nodes, x_pos) in enumerate(zip(layers, layer_positions)):
        for i in range(n_nodes):
            y_pos = 0.5 + (i - (n_nodes - 1) / 2) * 0.15
            color = (
                "lightblue"
                if l == 0
                else ("lightgreen" if l == len(layers) - 1 else "lightyellow")
            )
            circle = plt.Circle((x_pos, y_pos), 0.04, color=color, ec="black")
            ax.add_patch(circle)

    # Draw forward connections (blue)
    for l in range(len(layers) - 1):
        for i in range(layers[l]):
            for j in range(layers[l + 1]):
                y1 = 0.5 + (i - (layers[l] - 1) / 2) * 0.15
                y2 = 0.5 + (j - (layers[l + 1] - 1) / 2) * 0.15
                ax.annotate(
                    "",
                    xy=(layer_positions[l + 1] - 0.04, y2),
                    xytext=(layer_positions[l] + 0.04, y1),
                    arrowprops=dict(arrowstyle="->", color="blue", alpha=0.3),
                )

    # Labels
    ax.text(0.1, 0.9, "Input\nLayer", ha="center", fontsize=12)
    ax.text(0.35, 0.9, "Hidden\nLayer 1", ha="center", fontsize=12)
    ax.text(0.6, 0.9, "Hidden\nLayer 2", ha="center", fontsize=12)
    ax.text(0.85, 0.9, "Output\nLayer", ha="center", fontsize=12)

    # Forward/Backward arrows
    ax.annotate(
        "",
        xy=(0.75, 0.1),
        xytext=(0.2, 0.1),
        arrowprops=dict(arrowstyle="->", color="blue", lw=2),
    )
    ax.text(0.45, 0.12, "Forward Pass", ha="center", fontsize=11, color="blue")

    ax.annotate(
        "",
        xy=(0.2, 0.02),
        xytext=(0.75, 0.02),
        arrowprops=dict(arrowstyle="->", color="red", lw=2),
    )
    ax.text(
        0.45, 0.04, "Backward Pass (Gradients)", ha="center", fontsize=11, color="red"
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("Forward and Backward Propagation", fontsize=14)
    plt.show()


visualize_backprop()

## 6. Effect of Network Architecture

In [ ]:
# Compare different architectures
architectures = [
    [2, 4, 1],  # Shallow
    [2, 16, 1],  # Wider
    [2, 8, 8, 1],  # Deeper
    [2, 16, 8, 4, 1],  # Deep and wide
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, arch in enumerate(architectures):
    nn = NeuralNetwork(layer_sizes=arch, activation="relu", learning_rate=0.1)
    nn.fit(X_train_scaled, y_train, epochs=500, verbose=False)

    test_acc = np.mean(nn.predict(X_test_scaled) == y_test)

    # Plot decision boundary
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
    Z = nn.forward(grid).reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, levels=20, cmap="RdYlBu", alpha=0.8)
    axes[idx].scatter(X[y == 0, 0], X[y == 0, 1], c="blue", s=20, alpha=0.5)
    axes[idx].scatter(X[y == 1, 0], X[y == 1, 1], c="red", s=20, alpha=0.5)
    axes[idx].set_title(f"Architecture: {arch}\nTest Acc: {test_acc:.3f}")

plt.tight_layout()
plt.show()

## 7. Effect of Learning Rate

In [ ]:
# Compare learning rates
learning_rates = [0.001, 0.01, 0.1, 1.0]

plt.figure(figsize=(12, 5))

for lr in learning_rates:
    nn = NeuralNetwork(layer_sizes=[2, 16, 8, 1], activation="relu", learning_rate=lr)
    nn.fit(X_train_scaled, y_train, epochs=500, verbose=False)
    plt.plot(nn.loss_history, label=f"LR = {lr}")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Effect of Learning Rate on Training")
plt.legend()
plt.grid(True)
plt.show()

## 8. Key Takeaways

1. **Neural networks** are composed of layers of neurons (perceptrons)
2. **Activation functions** introduce non-linearity (ReLU is most common)
3. **Forward propagation** computes predictions layer by layer
4. **Backpropagation** computes gradients using the chain rule
5. **Learning rate** controls step size - too small is slow, too large is unstable
6. **Architecture** (depth and width) affects capacity to learn complex patterns
7. **Weight initialization** (He, Xavier) is crucial for training stability

In [ ]:
# Summary table
import pandas as pd

summary = pd.DataFrame(
    {
        "Concept": [
            "Perceptron",
            "Activation",
            "Forward Pass",
            "Backpropagation",
            "Loss Function",
        ],
        "Purpose": [
            "Basic unit",
            "Non-linearity",
            "Compute output",
            "Compute gradients",
            "Measure error",
        ],
        "Key Formula": [
            "y = σ(w·x + b)",
            "ReLU: max(0, x)",
            "a[l] = σ(W[l]·a[l-1] + b[l])",
            "∂L/∂W via chain rule",
            "Cross-entropy, MSE",
        ],
    }
)
print(summary.to_string(index=False))